# What author metadata the Semantic Scholar API gives

Could the socials use the Semantic Scholar API instead of OpenAlex? For a random sample of the researchers with `factuality_author = True`, how often each field the metrics need is filled. The public API is rate-limited, so the calls retry on 429 (takes a few minutes).

In [1]:
import json
import time
import urllib.error
import urllib.request

import pandas as pd

R = "../../../../results"
API = "https://api.semanticscholar.org/graph/v1"
N = 500   # researchers in the sample

ids = (pd.read_csv(f"{R}/6_factuality/factual_authors.csv", usecols=["researcher_id"])["researcher_id"]
         .dropna().astype("int64").drop_duplicates().sample(N, random_state=7).astype(str).tolist())


def call(path, body=None):
    for attempt in range(30):
        try:
            req = urllib.request.Request(f"{API}{path}", method="POST" if body else "GET",
                                         data=json.dumps(body).encode() if body else None,
                                         headers={"Content-Type": "application/json"})
            return json.load(urllib.request.urlopen(req, timeout=120))
        except urllib.error.HTTPError as e:
            if e.code != 429:
                raise
            time.sleep(3 * (attempt + 1))


def batch(fields, ids, size=100):
    out = []
    for i in range(0, len(ids), size):
        out += call(f"/author/batch?fields={fields}", {"ids": ids[i:i + size]})
        time.sleep(2)
    return out

## 1. Author fields

In [2]:
authors = batch("name,affiliations,homepage,paperCount,citationCount,hIndex,externalIds", ids)
found = [a for a in authors if a]
pd.Series({
    "researchers asked": N,
    "returned by the batch endpoint": len(found),
    "affiliations (institution)": sum(bool(a.get("affiliations")) for a in found),
    "homepage": sum(bool(a.get("homepage")) for a in found),
    "ORCID": sum("ORCID" in (a.get("externalIds") or {}) for a in found),
    "paperCount": sum(a.get("paperCount") is not None for a in found),
    "citationCount": sum(a.get("citationCount") is not None for a in found),
    "hIndex": sum(a.get("hIndex") is not None for a in found),
})

researchers asked                 500
returned by the batch endpoint    405
affiliations (institution)          1
homepage                            1
ORCID                               1
paperCount                        405
citationCount                     405
hIndex                            405
dtype: int64

The ones the batch endpoint does not return were merged by Semantic Scholar into another profile (a single `GET /author/{id}` follows the redirect).

## 2. Paper fields (for i10 / e-index, first and last year, coauthors)
The batch endpoint returns about 5,000 papers per answer in total, so researchers with many papers come back truncated.

In [3]:
papers = [a for a in batch("paperCount,papers.year,papers.citationCount,papers.authors", ids[:100]) if a]
rows = [{"paperCount": a["paperCount"], "returned": len(a.get("papers") or [])} for a in papers]
t = pd.DataFrame(rows)
all_p = [p for a in papers for p in a.get("papers") or []]
pd.Series({
    "researchers": len(t),
    "with all their papers returned": (t["returned"] >= t["paperCount"]).mean().round(3),
    "papers returned": len(all_p),
    "with year": sum(p.get("year") is not None for p in all_p),
    "with citationCount": sum(p.get("citationCount") is not None for p in all_p),
    "with coauthor ids": sum(bool(p.get("authors")) for p in all_p),
})

researchers                         85.000
with all their papers returned       0.953
papers returned                   3423.000
with year                         3406.000
with citationCount                3423.000
with coauthor ids                 3423.000
dtype: float64

## 3. Affiliations of the authors of a paper
The other place an institution could come from.

In [4]:
rows = []
for rid in ids[:10]:
    pid = (call(f"/author/{rid}/papers?fields=title&limit=1") or {}).get("data", [{}])
    if pid and pid[0].get("paperId"):
        auth = call(f"/paper/{pid[0]['paperId']}/authors?fields=affiliations").get("data", [])
        rows.append({"authors": len(auth), "with affiliations": sum(bool(a.get("affiliations")) for a in auth)})
    time.sleep(2)
pd.DataFrame(rows).sum()

authors              50
with affiliations     0
dtype: int64